# Notebook 20: Capstone — Semantic Drift Monitor (`drift-cam`)

## Why This Matters
This capstone assembles the tools from the series into a production-grade prototype: a semantic drift monitor for image streams. The system continuously monitors whether images arriving in production have drifted from the distribution the retrieval system was calibrated on — using CLIP embeddings, sliding window statistics, and automated alerting.

This is the `drift-cam` prototype — the tool from the BACKLOG.

## Structure
1. System architecture overview
2. Reference distribution encoding
3. Streaming encoder with CLIP
4. Statistical drift tests (KS, MMD)
5. Alert threshold calibration
6. End-to-end demo: detect drift in a simulated production stream
7. Production considerations

## What You'll Learn
- How to combine CLIP embeddings with sliding window drift tests
- How to calibrate alert thresholds to minimize false positives
- What a production-ready drift monitor architecture looks like
- Where to take this prototype next


## Architecture

```
Production images
       │
       ▼
  CLIP ViT-B/32 ──► embedding (512d)
       │
       ▼
  Sliding window buffer (W images)
       │
       ├── KS test vs reference distribution (per PCA component)
       │
       ├── MMD vs reference distribution
       │
       └── Centroid distance from reference mean
                    │
                    ▼
             Alert if statistic > threshold
                    │
                    ▼
         Log + notify + trigger re-evaluation
```

**Reference distribution:** encode your calibration set (the images used to validate your retrieval system) with CLIP. Store the PCA projection and reference statistics.

**Sliding window:** buffer the last W=100 production embeddings. Run drift tests comparing the window distribution to the reference distribution. Step every S=10 images.

**Alert thresholds:** calibrated from permutation tests on the reference distribution to achieve < 5% false positive rate.


In [ ]:
# %pip install -q openai-clip torch torchvision numpy scipy scikit-learn matplotlib

In [ ]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import clip
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import rbf_kernel
from torch.utils.data import DataLoader

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print(f"drift-cam v0.1 — CLIP ViT-B/32 on {device}")
print(f"Ready to monitor image streams.")

## 1. Reference Distribution

In [ ]:
# Reference: CIFAR-10 test set, first 5 classes (airplane, automobile, bird, cat, deer)
# Simulates a production system calibrated on a specific product category
cifar_test = torchvision.datasets.CIFAR10("/tmp/cifar10", train=False, download=True,
                                           transform=preprocess)
ref_indices = [i for i, (_, l) in enumerate(cifar_test) if l < 5][:500]
ref_subset  = torch.utils.data.Subset(cifar_test, ref_indices)
ref_loader  = DataLoader(ref_subset, batch_size=256, shuffle=False, num_workers=0)

print(f"Reference distribution: {len(ref_indices)} images (classes 0-4)")
print("Encoding reference...")
ref_embs = []
with torch.no_grad():
    for imgs, _ in ref_loader:
        feats = clip_model.encode_image(imgs.to(device))
        feats = feats / feats.norm(dim=-1, keepdim=True)
        ref_embs.append(feats.cpu().numpy())
ref_embs = np.vstack(ref_embs)
print(f"Reference embeddings: {ref_embs.shape}")

# Fit PCA on reference
N_PCA_COMPONENTS = 20
pca = PCA(n_components=N_PCA_COMPONENTS)
ref_pca = pca.fit_transform(ref_embs)
print(f"PCA fitted: {N_PCA_COMPONENTS} components, "
      f"{pca.explained_variance_ratio_[:5].sum():.1%} variance in top 5")

## 2. Threshold Calibration via Permutation Test

In [ ]:
def ks_score(ref_pca: np.ndarray, window_pca: np.ndarray) -> float:
    """Max KS statistic across PCA components."""
    return max(stats.ks_2samp(ref_pca[:, i], window_pca[:, i]).statistic
               for i in range(ref_pca.shape[1]))


def mmd_score(X: np.ndarray, Y: np.ndarray) -> float:
    gamma = 1.0 / np.median(np.sum((X[:50, None] - Y[None, :50])**2, axis=-1) + 1e-8)
    Kxx = rbf_kernel(X, X, gamma=gamma)
    Kyy = rbf_kernel(Y, Y, gamma=gamma)
    Kxy = rbf_kernel(X, Y, gamma=gamma)
    m, n = len(X), len(Y)
    return float(max(0, (Kxx.sum()-Kxx.diagonal().sum())/(m*(m-1)) +
                        (Kyy.sum()-Kyy.diagonal().sum())/(n*(n-1)) - 2*Kxy.mean()))


# Calibrate: compute KS and MMD on reference-vs-reference splits (null distribution)
print("Calibrating thresholds (100 permutation tests)...")
WINDOW_SIZE = 50
null_ks, null_mmd = [], []
for _ in range(100):
    idx_ref = np.random.choice(len(ref_pca), WINDOW_SIZE, replace=False)
    idx_win = np.random.choice(len(ref_pca), WINDOW_SIZE, replace=False)
    null_ks.append(ks_score(ref_pca[idx_ref], ref_pca[idx_win]))
    null_mmd.append(mmd_score(ref_pca[idx_ref], ref_pca[idx_win]))

# 95th percentile = 5% false positive rate threshold
KS_THRESHOLD  = np.percentile(null_ks,  95)
MMD_THRESHOLD = np.percentile(null_mmd, 95)

print(f"KS  threshold (95th pct): {KS_THRESHOLD:.4f}")
print(f"MMD threshold (95th pct): {MMD_THRESHOLD:.6f}")

## 3. Production Stream Simulation

In [ ]:
# Simulate a production stream: first 200 images in-distribution, then 200 drifted
# Drifted = classes 5-9 (dog, frog, horse, ship, truck) — different visual domain

drift_indices = [i for i, (_, l) in enumerate(cifar_test) if l >= 5][:200]
drift_subset  = torch.utils.data.Subset(cifar_test, drift_indices)
drift_loader  = DataLoader(drift_subset, batch_size=256, shuffle=False, num_workers=0)

print("Encoding production stream (in-dist + drifted)...")
indist_embs, drifted_embs = [], []
with torch.no_grad():
    for imgs, _ in DataLoader(torch.utils.data.Subset(cifar_test, ref_indices[:200]),
                               batch_size=256, num_workers=0):
        feats = clip_model.encode_image(imgs.to(device))
        indist_embs.append((feats / feats.norm(dim=-1, keepdim=True)).cpu().numpy())
    for imgs, _ in drift_loader:
        feats = clip_model.encode_image(imgs.to(device))
        drifted_embs.append((feats / feats.norm(dim=-1, keepdim=True)).cpu().numpy())

stream_embs = np.vstack([np.vstack(indist_embs), np.vstack(drifted_embs)])
stream_pca  = pca.transform(stream_embs)
DRIFT_START = len(indist_embs[0]) if len(indist_embs) == 1 else sum(x.shape[0] for x in indist_embs)
print(f"Stream: {len(stream_embs)} images ({DRIFT_START} in-dist, {len(stream_embs)-DRIFT_START} drifted)")

In [ ]:
# Run sliding window monitor
STEP = 10
timestamps, ks_scores, mmd_scores, alerts = [], [], [], []

for start in range(0, len(stream_pca) - WINDOW_SIZE + 1, STEP):
    window = stream_pca[start:start + WINDOW_SIZE]
    ref_sample = ref_pca[np.random.choice(len(ref_pca), WINDOW_SIZE, replace=False)]

    ks  = ks_score(ref_sample, window)
    mmd = mmd_score(ref_sample, window)
    alert = (ks > KS_THRESHOLD) or (mmd > MMD_THRESHOLD)

    timestamps.append(start + WINDOW_SIZE)
    ks_scores.append(ks)
    mmd_scores.append(mmd)
    alerts.append(alert)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

for ax, scores, threshold, name in [
    (axes[0], ks_scores,  KS_THRESHOLD,  "KS statistic"),
    (axes[1], mmd_scores, MMD_THRESHOLD, "MMD"),
]:
    ax.plot(timestamps, scores, color='royalblue', linewidth=2)
    ax.axhline(threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.4f})')
    ax.axvline(DRIFT_START, color='orange', linestyle=':', linewidth=2, label='Drift starts')
    # Shade alert regions
    for t, a in zip(timestamps, alerts):
        if a:
            ax.axvspan(t-STEP, t, alpha=0.15, color='red')
    ax.set_ylabel(name); ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)

axes[1].set_xlabel("Stream position (images processed)")
fig.suptitle("drift-cam: Semantic Drift Monitor — CLIP ViT-B/32", fontsize=13)
plt.tight_layout(); plt.show()

n_alerts = sum(alerts)
first_alert = timestamps[next(i for i, a in enumerate(alerts) if a)] if any(alerts) else None
print(f"Total alerts: {n_alerts}/{len(alerts)} windows")
print(f"First alert at stream position: {first_alert}")
print(f"Drift started at position: {DRIFT_START}")
if first_alert:
    print(f"Detection lag: {first_alert - DRIFT_START} images")

## 4. Production Considerations

**Embedding model:**
- CLIP ViT-B/32 is a good default; DINOv2 may give better sensitivity for visual distribution shift
- Lock the model version — a model update invalidates all reference statistics

**Window size:**
- Larger window = more statistical power, more lag
- W=50–200 is typical; calibrate for your expected shift magnitude and acceptable detection lag

**Step size:**
- Smaller step = more frequent checks, more compute
- S=10–50 is typical; compute cost is O(W) per step for KS, O(W²) for MMD

**PCA:**
- Fit on reference only — never refit on production data (that would track the drift, not detect it)
- 10–50 components captures most variance while keeping tests tractable

**Alert routing:**
- Page on sustained alerts (3+ consecutive windows), not single windows
- Log all statistics to a time-series store (Prometheus, InfluxDB) for retrospective analysis
- Trigger a retrieval recall@k re-evaluation when an alert fires

**What to do on drift detection:**
1. Check recent query logs — are users asking new things?
2. Check corpus additions — new document types added?
3. Re-run retrieval eval on recent queries
4. If recall degraded: reindex with a better model or add domain fine-tuning


## Series Summary

This series built the complete representation learning stack, from fundamentals to production:

| Notebook | Key tool/concept | Production relevance |
|----------|-----------------|---------------------|
| 01 | FastText | Multilingual, edge, high-throughput |
| 02 | Sentence-transformers | Universal embedding baseline |
| 03 | Fine-tuning embeddings | Domain adaptation |
| 04 | SimCLR | Contrastive learning foundations |
| 05 | MoCo | Large-scale contrastive learning |
| 06 | DINO | Dense visual features |
| 07 | CLIP | Multimodal retrieval |
| 08–09 | Probing + evaluation | Model selection, benchmarking |
| 10–11 | PCA + FAISS | Compression, fast search |
| 12–13 | Chroma + RAG | Production retrieval pipeline |
| 14 | Drift detection | Production monitoring |
| 15–17 | Cross-modal, fingerprinting, label noise | Advanced applications |
| 18–19 | Tabular + time series | Structured data representations |
| 20 | drift-cam prototype | End-to-end production system |

**Next series:** Series 3 — ML Security. How to attack and defend the models you've just learned to build.
